# 05 — Conformal Prediction

**Rift: Graph ML for Fraud Detection, Replay, and Audit**

This notebook demonstrates conformal prediction for uncertainty-aware fraud triage: instead of a binary label, each transaction gets a confidence band (high_confidence_fraud, review_needed, high_confidence_legit).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AngelP17/Rift/blob/main/notebooks/05_conformal.ipynb)


## Environment Setup


In [ ]:
# Clone repo and install dependencies (run once)
import subprocess, sys, os

REPO = "https://github.com/AngelP17/Rift.git"
if not os.path.exists("/content/Rift"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/Rift"], check=True)

os.chdir("/content/Rift")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "polars", "numpy", "pandas", "scikit-learn", "xgboost", "duckdb",
    "shap", "structlog", "python-dotenv", "rich", "jinja2", "pyarrow"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "torch", "--index-url", "https://download.pytorch.org/whl/cpu"], check=False)
sys.path.insert(0, "src")

try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    device = "cpu"
print(f"Device: {device}")


## Prepare Calibrated Scores


In [ ]:
import numpy as np
from data.generator import generate_transactions
from features.engine import build_features, get_feature_matrix
from data.splits import split_data
from models.baseline_xgb import TabularXGBoost
from models.calibrate import IsotonicCalibrator

df = generate_transactions(n_txns=20_000, fraud_rate=0.03, seed=42)
df_feat = build_features(df)
splits = split_data(df_feat, strategy="temporal")

X_train = get_feature_matrix(splits["train"]).to_numpy().astype(np.float32)
y_train = splits["train"]["is_fraud"].to_numpy().astype(np.float32)
X_val = get_feature_matrix(splits["val"]).to_numpy().astype(np.float32)
y_val = splits["val"]["is_fraud"].to_numpy().astype(np.float32)
X_test = get_feature_matrix(splits["test"]).to_numpy().astype(np.float32)
y_test = splits["test"]["is_fraud"].to_numpy().astype(np.float32)

model = TabularXGBoost(num_rounds=200)
model.fit(X_train, y_train, X_val, y_val)

val_raw = model.predict_proba(X_val)
calibrator = IsotonicCalibrator().fit(val_raw, y_val)

test_raw = model.predict_proba(X_test)
calibrated = calibrator.calibrate(test_raw)
print(f"Calibrated scores ready: {len(calibrated)} samples")


## Fit Conformal Predictor

The conformal predictor computes a nonconformity quantile from calibration residuals. At alpha=0.05, the prediction set covers the true label at least 95% of the time.


In [ ]:
from models.conformal import ConformalPredictor, compute_conformal_metrics

cp = ConformalPredictor(alpha=0.05)
cp.fit(calibrated, y_test)
print(f"Conformal quantile (q_hat): {cp.q_hat:.4f}")

bands = cp.predict_bands(calibrated)
metrics = compute_conformal_metrics(bands, y_test)

print(f"\n=== Conformal Metrics ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")


## Band Distribution


In [ ]:
import numpy as np

unique, counts = np.unique(bands, return_counts=True)
print("=== Decision Band Distribution ===")
for band, count in zip(unique, counts):
    pct = count / len(bands) * 100
    print(f"  {band}: {count:,} ({pct:.1f}%)")

print(f"\nReview rate: {metrics['review_rate']:.1%}")
print(f"Coverage: {metrics['empirical_coverage']:.1%} (target: 95%)")
print(f"Average set size: {metrics['average_set_size']:.2f} (target: < 1.4)")


## Compare: Hard Labels vs Conformal Triage


In [ ]:
hard_labels = (calibrated >= 0.5).astype(int)
hard_correct = (hard_labels == y_test).sum()
hard_accuracy = hard_correct / len(y_test)

band_correct = 0
for b, y in zip(bands, y_test):
    if b == "high_confidence_fraud" and y == 1:
        band_correct += 1
    elif b == "high_confidence_legit" and y == 0:
        band_correct += 1
    elif b == "review_needed":
        band_correct += 1  # review catches uncertain cases

band_accuracy = band_correct / len(y_test)

print("=== Hard Label vs Conformal Triage ===")
print(f"Hard label accuracy: {hard_accuracy:.4f}")
print(f"Conformal coverage:  {band_accuracy:.4f}")
print(f"\nConformal advantage: uncertain cases go to review instead of wrong decisions")


## Prediction Details


In [ ]:
results = cp.predict(calibrated[:10])
for i, r in enumerate(results):
    label = "FRAUD" if y_test[i] == 1 else "LEGIT"
    print(f"  Sample {i}: score={r['calibrated_score']:.3f} "
          f"band={r['confidence_band']:25s} "
          f"interval=[{r['interval_low']:.3f}, {r['interval_high']:.3f}] "
          f"actual={label}")
